In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cvxpy as cp
import os

In [2]:
etf_feed = [
  "QLD",
  "BNDW",
  "SH"
]



In [3]:
class DataStore:
  def __init__(self, debug:bool=False, **kwargs):
    super().__init__(
        debug=debug,
        **kwargs
    )
    self.debug = debug

  def download(self, indices:str, start:str, end:str, interval:str):
    data = {}
    for ticker in indices:
        clean_ticker = ticker.strip()
        try:
            tmp_data = yf.Ticker(clean_ticker) \
                          .history(
                              start=start,
                              end=end,
                              interval=interval
                          )["Close"]

            if tmp_data.empty:
                print(f"Warning: No data returned for {clean_ticker}")
                continue
            tmp_data.index = pd.to_datetime(tmp_data.index)\
                                    .tz_localize(None)\
                                    .normalize()
            tmp_data = tmp_data[~tmp_data.index.duplicated(keep='last')]
            data[clean_ticker] = tmp_data
        except Exception as e:
            print(f"Failed to download {clean_ticker}: {e}")

    if not data:
        raise ValueError(
            "No data was successfully downloaded  \
              for any of the provided tickers."
        )
    df = pd.concat(data, axis=1)
    return df.sort_index()

  def get_data(
      self,
      tickers:list,
      start:str,
      end:str,
      interval:str="1d",
      benchmark:str="^GSPC"
  ):
    self.benchmark_ticker = benchmark
    df_path = f"portfolio_{start}_{end}.parquet"
    benchmark_path = f"benchmark_{start}_{end}.parquet"

    if not os.path.exists(df_path) or not os.path.exists(benchmark_path):
      df = self.download(tickers, start, end, interval)
      bench_data = self.download([benchmark], start, end, interval)

      df.to_parquet(df_path)
      bench_data.to_parquet(benchmark_path)

    else:
      df = pd.read_parquet(df_path)
      bench_data= pd.read_parquet(benchmark_path)

    returns_raw = df.pct_change().dropna()
    benchmark = bench_data.pct_change().dropna()
    self.universe = returns_raw.columns

    return returns_raw, benchmark

  def plot_data(self):
    (np.cumsum(self.returns_raw * 100, axis=0) + 100).plot(figsize=(15, 10))
    plt.show()

  def plot_benchmark(self):
    (np.cumsum(self.benchmark * 100, axis=0) + 100).plot(figsize=(15, 10))
    plt.show()

In [4]:
class GeomRegimeDetector:
  def __init__(self, debug, **kwargs):
    super().__init__(
        debug=debug,
        **kwargs
    )
    self.debug = debug


  def get_regime(self, mkt_returns):
    pass

In [5]:
class Optimizer:
  def __init__(self, debug, **kwargs):
    super().__init__(
        debug=debug,
        **kwargs
    )
    self.debug = debug

  def optimize(self, returns, risk_aversion, orth_penalty):
    pass




In [6]:
class PortfolioBuilder(Optimizer, DataStore, GeomRegimeDetector):
  def __init__(self, debug, **kwargs):
    super().__init__(
        debug=debug,
        **kwargs
    )
    self.debug = debug

  def CAPM(self, X, y):
    betas = {}
    for col in y.columns:
      cov = np.cov(X, y[col])
      beta = cov[0, 1] / cov[1, 1]
      betas[col] = beta

    return pd.Series(betas)

  def transform_data(data, benchmark):
    returns = data.pct_change()
    mkt_returns = benchmark.pct_change()

    betas = self.CAPM(mkt_returns, returns)
    regime = self.get_regime(mkt_returns)

    return returns, mkt_returns


